# TICKET-002 — Data Quality Assessment
Goal: verify that the numbers are internally consistent and valid before cleaning.
No data is modified in this notebook.

**Data versions:** `82610ENG` (CBS, updated 19 June 2026) and `84575ENG` (CBS, updated 30 September 2026), downloaded October 2026.
All findings refer to these versions. CBS will publish corrections in November 2026 (see Findings 1 and 7).

## 0. Setup — load raw files and classify periods (temporary, not saved)

In [0]:
from pyspark.sql import functions as F

annual_raw  = spark.read.option("header", True).option("inferSchema", True) \
    .csv("/Volumes/workspace/energy_project/raw/renewable_electricity_1990_latest.csv")
monthly_raw = spark.read.option("header", True).option("inferSchema", True) \
    .csv("/Volumes/workspace/energy_project/raw/renewable_electricity_monthly.csv")

months = "January|February|March|April|May|June|July|August|September|October|November|December"

monthly = (monthly_raw
    .withColumn("Periods", F.trim("Periods"))
    .withColumn("year", F.substring("Periods", 1, 4).cast("int"))
    .withColumn("period_type",
        F.when(F.col("Periods").rlike("^[0-9]{4}$"), "year")
         .when(F.col("Periods").rlike("quarter"), "quarter")
         .when(F.col("Periods").rlike(months), "month")
         .otherwise("UNKNOWN")))

display(monthly.groupBy("period_type").count())

period_type,count
year,97
month,607
quarter,202


## 1. Hierarchy check — do components add up to their totals?
Checked for all three production measures (million kWh).
Nulls in components are treated as 0 (technology did not exist yet).
Tolerance: 1 million kWh (rounding).

In [0]:
prod_cols = ["GrossProductionWithNormalisation_1",
             "GrossProductionOfElectricity_2",
             "NetProductionOfElectricity_3"]

hierarchy = {
    "Wind energy, total": ["Onshore wind energy", "Offshore wind energy"],
    "Total biogas": ["Biogas from landfills", "Biogas from sewage water purification",
                     "Biogas, co-digestion of manure", "Other biogas"],
    "Total biomass": ["Total biogas", "Municipal waste; renewable fraction",
                      "Biomass boilers companies, CHP", "Co-firing of biomass in electr. plants"],
    "Total renewable energy sources": ["Hydropower", "Wind energy, total",
                                       "Solar photovoltaic", "Total biomass"],
}

results = []
for prod in prod_cols:
    wide = (annual_raw.groupBy("Periods")
            .pivot("EnergySourcesTechniques")
            .agg(F.first(prod)))

    for total, parts in hierarchy.items():
        parts_sum = sum(F.coalesce(F.col(f"`{p}`"), F.lit(0)) for p in parts)
        diff = wide.select("Periods", (F.col(f"`{total}`") - parts_sum).alias("diff"))

        stats = diff.agg(
            F.count(F.when(F.abs("diff") > 1, True)).alias("years_failing"),
            F.round(F.max(F.abs("diff")), 2).alias("max_abs_diff")
        ).collect()[0]

        failing_years = [r["Periods"] for r in diff.filter(F.abs("diff") > 1)
                                                  .orderBy("Periods").collect()]

        results.append((prod, total, len(parts),
                        stats["years_failing"], stats["max_abs_diff"],
                        ", ".join(str(y) for y in failing_years)))

display(spark.createDataFrame(results,
        ["column", "total", "n_components", "years_failing", "max_abs_diff", "failing_years"]))

column,total,n_components,years_failing,max_abs_diff,failing_years
GrossProductionWithNormalisation_1,"Wind energy, total",2,0,1.0,
GrossProductionWithNormalisation_1,Total biogas,4,0,1.0,
GrossProductionWithNormalisation_1,Total biomass,4,1,11.0,2014
GrossProductionWithNormalisation_1,Total renewable energy sources,4,0,1.0,
GrossProductionOfElectricity_2,"Wind energy, total",2,0,1.0,
GrossProductionOfElectricity_2,Total biogas,4,1,357.0,2023
GrossProductionOfElectricity_2,Total biomass,4,0,1.0,
GrossProductionOfElectricity_2,Total renewable energy sources,4,0,1.0,
NetProductionOfElectricity_3,"Wind energy, total",2,0,1.0,
NetProductionOfElectricity_3,Total biogas,4,1,348.0,2023


### 1a. Drill-down — compare production measures for biomass categories (2014, 2023)
Hypothesis tested: `_1` (with normalisation) equals `_2` for biomass, since normalisation applies to wind and hydro only.
**Result: rejected** — `_1` differs from `_2` for several biomass categories (see Finding 2).

In [0]:
biomass_cats = hierarchy["Total biogas"] + ["Total biogas"] + hierarchy["Total biomass"][1:] + ["Total biomass"]

display(annual_raw
    .filter(F.col("Periods").isin(2014, 2023) &
            F.col("EnergySourcesTechniques").isin(biomass_cats))
    .select("Periods", "EnergySourcesTechniques",
            F.col("GrossProductionWithNormalisation_1").alias("gross_norm_1"),
            F.col("GrossProductionOfElectricity_2").alias("gross_2"),
            F.col("NetProductionOfElectricity_3").alias("net_3"))
    .withColumn("diff_1_minus_2", F.col("gross_norm_1") - F.col("gross_2"))
    .orderBy("Periods", "EnergySourcesTechniques"))

Periods,EnergySourcesTechniques,gross_norm_1,gross_2,net_3,diff_1_minus_2
2014,Biogas from landfills,55.0,46.0,46.0,9.0
2014,Biogas from sewage water purification,201.0,201.0,195.0,0.0
2014,"Biogas, co-digestion of manure",525.0,525.0,509.0,0.0
2014,"Biomass boilers companies, CHP",1166.0,1166.0,1021.0,0.0
2014,Co-firing of biomass in electr. plants,933.0,933.0,867.0,0.0
2014,Municipal waste; renewable fraction,1909.0,1909.0,1481.0,0.0
2014,Other biogas,296.0,233.0,226.0,63.0
2014,Total biogas,1077.0,1005.0,976.0,72.0
2014,Total biomass,5096.0,5013.0,4345.0,83.0
2023,Biogas from landfills,26.0,20.0,19.0,6.0


### 1b. Trend — Total biogas vs sum of components, 2020–2025 (all three measures)

In [0]:
cols3 = {"GrossProductionWithNormalisation_1": "norm_1",
         "GrossProductionOfElectricity_2": "gross_2",
         "NetProductionOfElectricity_3": "net_3"}
parts = hierarchy["Total biogas"]

base = annual_raw.filter(F.col("Periods").between(2020, 2025))

totals = (base.filter(F.col("EnergySourcesTechniques") == "Total biogas")
              .select("Periods", *[F.col(c).alias(f"total_{a}") for c, a in cols3.items()]))
sums = (base.filter(F.col("EnergySourcesTechniques").isin(parts))
            .groupBy("Periods")
            .agg(*[F.sum(c).alias(f"parts_{a}") for c, a in cols3.items()]))

display(totals.join(sums, "Periods").orderBy("Periods"))

Periods,total_norm_1,total_gross_2,total_net_3,parts_norm_1,parts_gross_2,parts_net_3
2020,1143.0,870.0,843.0,1144.0,870.0,843.0
2021,756.0,868.0,842.0,755.0,868.0,842.0
2022,390.0,840.0,815.0,389.0,840.0,814.0
2023,379.0,704.0,683.0,380.0,347.0,335.0
2024,339.0,514.0,498.0,339.0,514.0,498.0
2025,336.0,442.0,428.0,336.0,442.0,429.0


### 1c. Impact — size of the 2023 gap relative to total renewable production

In [0]:
y = 2023

def value(category):
    return (annual_raw.filter((F.col("Periods") == y) &
                              (F.col("EnergySourcesTechniques") == category))
                      .select("GrossProductionOfElectricity_2").collect()[0][0])

parts_sum = (annual_raw.filter((F.col("Periods") == y) &
                               F.col("EnergySourcesTechniques").isin(parts))
                       .agg(F.sum("GrossProductionOfElectricity_2")).collect()[0][0])

total_biogas = value("Total biogas")
total_ren    = value("Total renewable energy sources")
gap          = total_biogas - parts_sum

print(f"{y}: Total biogas = {total_biogas}, sum of components = {parts_sum}, gap = {gap}")
print(f"Gap as % of total renewable production ({total_ren}): {gap / total_ren * 100:.2f}%")

2023: Total biogas = 704.0, sum of components = 347.0, gap = 357.0
Gap as % of total renewable production (56952.0): 0.63%


### 1d. Root cause — are `_1` and `_2` identical for the affected components?

In [0]:
display(annual_raw
    .filter(F.col("Periods").between(2020, 2025) &
            F.col("EnergySourcesTechniques").isin("Biogas, co-digestion of manure", "Other biogas"))
    .select("Periods", "EnergySourcesTechniques",
            F.col("GrossProductionWithNormalisation_1").alias("norm_1"),
            F.col("GrossProductionOfElectricity_2").alias("gross_2"))
    .withColumn("identical", F.col("norm_1") == F.col("gross_2"))
    .orderBy("EnergySourcesTechniques", "Periods"))

Periods,EnergySourcesTechniques,norm_1,gross_2,identical
2020,"Biogas, co-digestion of manure",565.0,447.0,false
2021,"Biogas, co-digestion of manure",327.0,457.0,false
2022,"Biogas, co-digestion of manure",77.0,447.0,false
2023,"Biogas, co-digestion of manure",64.0,64.0,true
2024,"Biogas, co-digestion of manure",48.0,186.0,false
2025,"Biogas, co-digestion of manure",46.0,120.0,false
2020,Other biogas,340.0,204.0,false
2021,Other biogas,209.0,218.0,false
2022,Other biogas,108.0,216.0,false
2023,Other biogas,106.0,106.0,true


### Finding 1 — Data entry error: biogas components 2023 (confirmed by CBS)
- **Symptom:** in 2023 only, Total biogas ≠ sum of components in gross (704 vs 347) and net (683 vs 335) production. All other years reconcile within ±1.
- **Root cause:** for *Co-digestion of manure* (64) and *Other biogas* (106), the 2023 values without normalisation are identical to the values with normalisation. In every other year 2020–2025 the two measures differ substantially (e.g. co-digestion 2022: 77 vs 447). The normalised values were entered in the non-normalised column.
- **Which number is right:** Total biogas (704) follows a smooth trend (840 → 704 → 514) and is consistent with Total biomass, so the **total is trusted**. The two affected components should sum to ≈527 (split unknown).
- **Impact:** 357 million kWh ≈ 0.63% of total renewable production in 2023. Wind, solar and hydro are unaffected.
- **Decision:** keep totals; flag 2023 *Co-digestion* and *Other biogas* as unreliable; no imputation.
- **Confirmed by CBS Infoservice (26 Oct 2026, ref. #1697955):** wrong values were placed in the non-normalised column during the 2023 update. CBS will correct this in the November 2026 update.
- **Follow-up:** after the November update, re-download `82610ENG` and re-run Section 1. Expected: Total biogas 2023 reconciles.

### Finding 2 — Normalised column (`_1`) follows EU RED accounting (confirmed by CBS documentation)
Per CBS table notes, `_1` differs from `_2` in three ways:
1. **Weather correction** for wind (capacity × average production per MW over the past 5 years) and hydro (15 years). Solar is **not** normalised (EU convention).
2. **Green gas** (upgraded biogas injected into the gas grid) is included in `_1` only. This explains `_1 > _2` for some biogas categories (e.g. 2014).
3. **RED II sustainability criteria** (from reporting year 2021): non-compliant solid/gaseous biomass in large installations is excluded from `_1`. This explains the drop in `_1` biogas (1143 → 756 → 390, 2020–2022).
- **Implication for Finding 1:** exact equality of `_1` and `_2` for two components in one year only is very unlikely under these rules, which supports the data entry hypothesis.
- **Implication for analysis:** for wind, `_2 − _1` approximates actual minus expected-under-average-weather, but it can also include post-submission revisions (see Finding 7), so it is not a pure weather effect.
- **Minor:** Total biomass in `_1` does not reconcile in 2014 (diff 11 million kWh), possibly due to green gas allocated at total level. Not investigated further, since `_1` is not used for analysis.
- **Decision:** use `_2` as the main production measure.

**Known limitation:** the CSV was downloaded without statistical symbols, so empty cells ("not applicable" or "nil") and "." ("unknown/confidential") are both read as null and cannot be distinguished.

## 2. Time aggregation check — do months add up to quarters and years?
Only complete periods are compared (12 months per year, 3 per quarter). First pass with a fixed tolerance of 1 million kWh; see 2a for a rounding-aware tolerance.

In [0]:
cols = ["GrossProduction_1", "WindEnergyTotal_12", "WindEnergyOnShore_13",
        "WindEnergyOffShore_14", "SolarPhotovoltaic_15", "HydroPower_11", "Biomass_9"]

m = (monthly.filter(F.col("period_type") == "month")
     .withColumn("date", F.to_date("Periods", "yyyy MMMM"))
     .withColumn("quarter", F.quarter("date")))

print("Unparsed month labels:", m.filter(F.col("date").isNull()).count())   # expect 0

q = (monthly.filter(F.col("period_type") == "quarter")
     .withColumn("quarter", F.regexp_extract("Periods", r"(\d)(st|nd|rd|th) quarter", 1).cast("int")))
yr = monthly.filter(F.col("period_type") == "year")

def compare(keys, ref, n_expected, level):
    agg = (m.groupBy(*keys)
             .agg(F.count("*").alias("n_months"), *[F.sum(c).alias(c) for c in cols])
             .filter(F.col("n_months") == n_expected))
    joined = agg.alias("a").join(ref.alias("r"), keys)
    out = []
    for c in cols:
        d = joined.select(*keys, (F.col(f"a.{c}") - F.col(f"r.{c}")).alias("diff")).dropna()
        s = d.agg(F.count("*").alias("compared"),
                  F.count(F.when(F.abs("diff") > 1, True)).alias("failing"),
                  F.round(F.max(F.abs("diff")), 2).alias("max_abs_diff")).collect()[0]
        out.append((level, c, s["compared"], s["failing"], s["max_abs_diff"]))
    return out

results2 = compare(["year"], yr, 12, "months→year") + \
           compare(["year", "quarter"], q, 3, "months→quarter")

display(spark.createDataFrame(results2, ["level", "column", "compared", "failing", "max_abs_diff"]))

Unparsed month labels: 0


level,column,compared,failing,max_abs_diff
months→year,GrossProduction_1,50,15,118.0
months→year,WindEnergyTotal_12,24,2,2.0
months→year,WindEnergyOnShore_13,24,5,2.0
months→year,WindEnergyOffShore_14,20,3,3.0
months→year,SolarPhotovoltaic_15,11,1,2.0
months→year,HydroPower_11,11,2,2.0
months→year,Biomass_9,11,3,3.0
months→quarter,GrossProduction_1,202,2,9.0
months→quarter,WindEnergyTotal_12,98,0,1.0
months→quarter,WindEnergyOnShore_13,98,0,1.0


### 2a. Rounding-aware tolerance + drill-down on GrossProduction_1
Each published value is rounded to the nearest integer (±0.5), so the tolerance scales with the number of months summed: 6 for a year, 1.5 for a quarter.


In [0]:
def compare_tol(keys, ref, n_expected, level):
    tol = 0.5 * n_expected
    agg = (m.groupBy(*keys)
             .agg(F.count("*").alias("n_months"), *[F.sum(c).alias(c) for c in cols])
             .filter(F.col("n_months") == n_expected))
    joined = agg.alias("a").join(ref.alias("r"), keys)
    out = []
    for c in cols:
        d = joined.select(*keys, (F.col(f"a.{c}") - F.col(f"r.{c}")).alias("diff")).dropna()
        s = d.agg(F.count("*").alias("compared"),
                  F.count(F.when(F.abs("diff") > tol, True)).alias("failing"),
                  F.round(F.max(F.abs("diff")), 2).alias("max_abs_diff")).collect()[0]
        out.append((level, c, tol, s["compared"], s["failing"], s["max_abs_diff"]))
    return out

results2b = compare_tol(["year"], yr, 12, "months→year") + \
            compare_tol(["year", "quarter"], q, 3, "months→quarter")
display(spark.createDataFrame(results2b,
        ["level", "column", "tolerance", "compared", "failing", "max_abs_diff"]))

# Drill-down: which years fail for GrossProduction_1, and how big is the gap in %?
gp = (m.groupBy("year")
        .agg(F.count("*").alias("n_months"), F.sum("GrossProduction_1").alias("sum_months"))
        .filter("n_months = 12")
        .join(yr.select("year", F.col("GrossProduction_1").alias("annual")), "year")
        .withColumn("diff", F.col("sum_months") - F.col("annual"))
        .withColumn("diff_pct", F.round(F.col("diff") / F.col("annual") * 100, 3))
        .filter(F.abs("diff") > 6)
        .orderBy("year"))
display(gp)

level,column,tolerance,compared,failing,max_abs_diff
months→year,GrossProduction_1,6.0,50,15,118.0
months→year,WindEnergyTotal_12,6.0,24,0,2.0
months→year,WindEnergyOnShore_13,6.0,24,0,2.0
months→year,WindEnergyOffShore_14,6.0,20,0,3.0
months→year,SolarPhotovoltaic_15,6.0,11,0,2.0
months→year,HydroPower_11,6.0,11,0,2.0
months→year,Biomass_9,6.0,11,0,3.0
months→quarter,GrossProduction_1,1.5,202,2,9.0
months→quarter,WindEnergyTotal_12,1.5,98,0,1.0
months→quarter,WindEnergyOnShore_13,1.5,98,0,1.0


year,n_months,sum_months,annual,diff,diff_pct
1976,12,58138.0,58166.0,-28.0,-0.048
1977,12,58284.0,58313.0,-29.0,-0.05
1978,12,61595.0,61624.0,-29.0,-0.047
1979,12,64465.0,64492.0,-27.0,-0.042
1980,12,64806.0,64834.0,-28.0,-0.043
1981,12,64051.0,64081.0,-30.0,-0.047
1982,12,60312.0,60340.0,-28.0,-0.046
1983,12,59651.0,59677.0,-26.0,-0.044
1984,12,62780.0,62827.0,-47.0,-0.075
1985,12,62937.0,62975.0,-38.0,-0.06


In [0]:
display(m.groupBy("year", "quarter")
         .agg(F.count("*").alias("n"), F.sum("GrossProduction_1").alias("sum_months"))
         .filter("n = 3")
         .join(q.select("year", "quarter", F.col("GrossProduction_1").alias("quarterly")), ["year", "quarter"])
         .withColumn("diff", F.col("sum_months") - F.col("quarterly"))
         .filter(F.abs("diff") > 1.5).orderBy("year", "quarter"))

year,quarter,n,sum_months,quarterly,diff
1978,1,3,15955.0,15946.0,9.0
1990,3,3,17028.0,17026.0,2.0


### Findings — Section 2

**Finding 3 — Renewable columns are temporally consistent ✅**
- Months add up to quarters and years for wind (total, onshore, offshore), solar, hydro and biomass once the tolerance accounts for rounding.
- Each published value is rounded to an integer (±0.5), so a 12-month sum can deviate by up to ±6. With a fixed tolerance of 1, several years falsely failed (max diff 2–3).
- **Lesson:** tolerances must scale with the number of rounded values being summed.

**Finding 4 — GrossProduction_1: monthly ≠ annual before 1991**
- 15 failing years, all consecutive (1976–1990); from 1991 onward months reconcile with the annual total.
- The gap is systematic (≈ −28 million kWh in most years) and small (max 0.17%, in 1988).
- Quarterly level shows the same pattern: only 2 of 202 quarters fail (1978 Q1: +9, 1990 Q3: +2), both before 1991.
- Likely cause: different coverage between monthly and annual statistics in that period (not confirmed).
- **Decision:** no action. The project uses monthly data from 2002 (wind) and 2015 (other sources).

In [0]:
%pip install cbsodata

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import cbsodata

info = cbsodata.get_meta('82610ENG', 'TableInfos')[0]
print(info['Description'])

for p in cbsodata.get_meta('82610ENG', 'DataProperties'):
    if p.get('Description'):
        print(f"--- {p['Key']} ---\n{p['Description']}\n")

CONTENTS

1. General information
2. Definitions and explanation of symbols
3. Links to relevant tables and articles
4. Sources and methods
5. Further information


1. GENERAL INFORMATION

This table contains information about the Dutch production of renewable electricity, the number of installations used and the installed capacity of these installations. During production, a distinction is made between normalised gross production and non-standard gross and net production without normalisation.

Production of electricity is shown in million kilowatt hours and as a percentage of total electricity consumption in the Netherlands. The production of renewable electricity is compared with total electricity consumption and not against total electricity production. This choice is due to European conventions.

The data is broken down according to the type of energy source and the technique used to obtain the electricity. A distinction is made between four main categories: hydro power, wind energ

In [0]:
info = cbsodata.get_meta('84575ENG', 'TableInfos')[0]
print(info['Description'])

for p in cbsodata.get_meta('84575ENG', 'DataProperties'):
    if p.get('Description'):
        print(f"--- {p['Key']} ---\n{p['Description']}\n")

CONTENTS

1. General information
2. Definitions and explanation of symbols
3. Links to relevant tables and articles
4. Sources and methods
5. Further information


1. GENERAL INFORMATION 

This table shows the supply of electricity. Consumption of electricity is calculated from the supply variables. The supply of electricity primarily includes production plus imports minus exports. The majority of the electricity produced is supplied to the public electricity grid by, for example, power stations and wind turbines. A smaller part is generated by companies themselves for the benefit of their own business processes. For example, many greenhouse companies generate their own electricity for the lighting of their greenhouses. 

The net production is determined as gross production minus the own consumption of electricity. Own consumption is the amount of electricity that a producer or installation consumes during electricity production. The net production is broken down in this table into the

**Finding 5 — Monthly source columns are net production (CBS documentation)**
- In `84575ENG`, the breakdown by energy source (wind, solar, hydro, biomass) refers to **net** production; `GrossProduction_1` is the only gross total.
- **Implication:** cross-file comparisons must use annual `_3` (net), not `_2` (gross). Capacity factors from monthly data are net-based.
- Documentation confirms: monthly totals from 1976, full monthly breakdown from 2015, offshore wind from autumn 2006. Wind is in fact available monthly from 2002, earlier than documented.
- Documentation does not mention a methodology change before 1991, so the cause of Finding 4 remains unconfirmed.

## 3. Normalisation logic — where does `_1` differ from `_2`?
Per CBS documentation, `_1` differs from `_2` through (1) weather correction for wind and hydro, (2) green gas, (3) RED II sustainability criteria for biomass (from 2021).
**Testable prediction:** solar is affected by none of these, so `_1` must equal `_2` for solar in every year.

In [0]:
display(annual_raw
    .withColumn("diff", F.col("GrossProductionWithNormalisation_1") - F.col("GrossProductionOfElectricity_2"))
    .groupBy("EnergySourcesTechniques")
    .agg(F.count(F.when(F.abs("diff") > 1, True)).alias("years_different"),
         F.count("diff").alias("years_compared"),
         F.round(F.max(F.abs("diff")), 0).alias("max_abs_diff"))
    .orderBy(F.desc("years_different")))

EnergySourcesTechniques,years_different,years_compared,max_abs_diff
Total renewable energy sources,36,36,3112.0
Total biogas,36,36,450.0
Total biomass,36,36,1455.0
Biogas from landfills,36,36,19.0
Hydropower,34,36,44.0
Onshore wind energy,33,36,2125.0
"Wind energy, total",33,36,2651.0
Offshore wind energy,20,20,525.0
Other biogas,14,36,136.0
Biogas from sewage water purification,8,36,29.0


### 3a. Drill-down — which years differ, per category?

In [0]:
display(annual_raw
    .withColumn("diff", F.col("GrossProductionWithNormalisation_1") - F.col("GrossProductionOfElectricity_2"))
    .filter((F.abs("diff") > 1) &
            F.col("EnergySourcesTechniques").isin(
                "Solar photovoltaic", "Co-firing of biomass in electr. plants",
                "Biomass boilers companies, CHP", "Biogas, co-digestion of manure",
                "Biogas from sewage water purification", "Other biogas"))
    .groupBy("EnergySourcesTechniques")
    .agg(F.sort_array(F.collect_list(F.struct("Periods", F.round("diff", 0).alias("diff"))))
          .alias("year_diff")))

EnergySourcesTechniques,year_diff
Solar photovoltaic,"List(List(2020, 197.0))"
"Biogas, co-digestion of manure","List(List(2018, 27.0), List(2019, 70.0), List(2020, 118.0), List(2021, -130.0), List(2022, -370.0), List(2024, -138.0), List(2025, -74.0))"
Co-firing of biomass in electr. plants,"List(List(2024, -89.0))"
"Biomass boilers companies, CHP","List(List(2011, -12.0), List(2021, -491.0), List(2022, -1005.0), List(2023, -151.0), List(2024, -60.0), List(2025, -60.0))"
Biogas from sewage water purification,"List(List(2018, 3.0), List(2019, 5.0), List(2020, 13.0), List(2021, 20.0), List(2022, 21.0), List(2023, 27.0), List(2024, 26.0), List(2025, 29.0))"
Other biogas,"List(List(2011, 4.0), List(2012, 15.0), List(2013, 42.0), List(2014, 63.0), List(2015, 87.0), List(2016, 99.0), List(2017, 123.0), List(2018, 104.0), List(2019, 116.0), List(2020, 136.0), List(2021, -9.0), List(2022, -108.0), List(2024, -66.0), List(2025, -64.0))"


### 3b. Drill-down — Solar photovoltaic 2020 (`_1` ≠ `_2`; explained by CBS, see Finding 7)
Monthly file (`84575ENG`, net) is used as an independent reference.

In [0]:
solar_annual = (annual_raw
    .filter((F.col("EnergySourcesTechniques") == "Solar photovoltaic") & F.col("Periods").between(2017, 2023))
    .select(F.col("Periods").alias("year"),
            F.col("GrossProductionWithNormalisation_1").alias("norm_1"),
            F.col("GrossProductionOfElectricity_2").alias("gross_2"),
            F.col("NetProductionOfElectricity_3").alias("net_3"),
            F.col("ElectricalCapacityEndOfYear_8").alias("capacity_MW")))

solar_monthly = yr.select("year", F.col("SolarPhotovoltaic_15").alias("monthly_file_net"))

display(solar_annual.join(solar_monthly, "year", "left")
        .withColumn("diff_1_2", F.col("norm_1") - F.col("gross_2"))
        .withColumn("gross_2_per_MW", F.round(F.col("gross_2") / F.col("capacity_MW"), 3))
        .orderBy("year"))

year,norm_1,gross_2,net_3,capacity_MW,monthly_file_net,diff_1_2,gross_2_per_MW
2017,2208.0,2208.0,2208.0,2911.0,2204.0,0.0,0.759
2018,3709.0,3709.0,3709.0,4608.0,3708.0,0.0,0.805
2019,5399.0,5399.0,5399.0,7226.0,5399.0,0.0,0.747
2020,8765.0,8568.0,8568.0,11108.0,8567.0,197.0,0.771
2021,11304.0,11304.0,11304.0,14823.0,11304.0,0.0,0.763
2022,16657.0,16657.0,16657.0,17356.0,16657.0,0.0,0.96
2023,19607.0,19607.0,19607.0,21957.0,19607.0,0.0,0.893


### Findings — Section 3

**Finding 6 — CBS documentation confirmed by the data ✅**
- Wind and hydro: `_1 ≠ _2` in almost every year (weather correction). Municipal waste: `_1 = _2` in every year.
- Biomass sign flip in 2021: before 2021 `_1 > _2` (green gas added), from 2021 `_1 < _2` (RED II exclusion), for co-digestion, other biogas and biomass boilers. This matches the documented start of RED II criteria (reporting year 2021).
- **Additional evidence for Finding 1:** co-digestion of manure differs between `_1` and `_2` in every year 2018–2025 except 2023; other biogas in every year 2011–2025 except 2023.


**Finding 7 — Solar 2020: `_1` ≠ `_2` is intentional (explained by CBS)**
- Solar is not normalised, yet `_1` = 8765 vs `_2` = 8568 in 2020 (+197, +2.3%). `_2`, `_3` and the monthly file agree (8568 / 8568 / 8567).
- **CBS explanation (27 Oct 2026, ref. #1697955):** `_1` is the production reported under the EU Renewable Energy Directive (RED). Once officially submitted to Eurostat, these values are **frozen**. Better solar data became available after the 2020 submission, so CBS revised `_2`/`_3` but kept the submitted RED 1 value in `_1`. The same will apply to 2021–2024 (calculated under RED 2, now that RED 3 is in force). CBS will document this in the table notes in November 2026.
- **Lesson:** an internal inconsistency is not necessarily an error. It can reflect a reporting convention, so domain context is needed before labelling it.
- **Implications:**
  1. `_2` is the right measure for analysis: it incorporates revisions, while `_1` does not.
  2. `_2 − _1` for wind is not a pure weather effect: it can also include post-submission revisions. It must be used with caution in the weather analysis.
  3. After the November update, `_1 ≠ _2` is expected for solar in 2021–2024. This is not a new error.

## 4. Implied consumption — is the % column consistent with production?
If `_5 = _2 / consumption × 100`, then `_2 / _5 × 100` should give the same consumption for every source in a given year.
Only rows with `_5 ≥ 5%` are used: `_5` is rounded to one decimal, so small shares carry large rounding error.

In [0]:
ic = (annual_raw
    .filter(F.col("GrossProductionOfElectricity_5") >= 5)
    .withColumn("implied_consumption",
                F.col("GrossProductionOfElectricity_2") / F.col("GrossProductionOfElectricity_5") * 100))

per_year = (ic.groupBy("Periods")
    .agg(F.count("*").alias("n_sources"),
         F.collect_list("EnergySourcesTechniques").alias("sources"),
         F.round(F.min("implied_consumption")).alias("min_consumption"),
         F.round(F.max("implied_consumption")).alias("max_consumption"),
         F.round((F.max("implied_consumption") - F.min("implied_consumption"))
                 / F.avg("implied_consumption") * 100, 2).alias("spread_pct"))
    .orderBy("Periods"))

# Compare with net consumption from the monthly file (annual rows)
per_year = per_year.join(
    yr.select(F.col("year").alias("Periods"),
              F.col("NetConsumptionCalculated_30").alias("net_consumption_84575")),
    "Periods", "left"
).withColumn("ratio_to_84575",
             F.round(F.col("max_consumption") / F.col("net_consumption_84575"), 3))

display(per_year.orderBy("Periods"))

Periods,n_sources,sources,min_consumption,max_consumption,spread_pct,net_consumption_84575,ratio_to_84575
2005,1,List(Total renewable energy sources),118286.0,118286.0,0.0,110301.0,1.072
2006,1,List(Total renewable energy sources),120314.0,120314.0,0.0,111804.0,1.076
2007,1,List(Total renewable energy sources),122718.0,122718.0,0.0,114027.0,1.076
2008,1,List(Total renewable energy sources),123484.0,123484.0,0.0,115266.0,1.071
2009,2,"List(Total renewable energy sources, Total biomass)",118545.0,118660.0,0.1,109736.0,1.081
2010,2,"List(Total renewable energy sources, Total biomass)",122062.0,122094.0,0.03,112681.0,1.084
2011,2,"List(Total renewable energy sources, Total biomass)",122979.0,123077.0,0.08,113555.0,1.084
2012,2,"List(Total renewable energy sources, Total biomass)",120468.0,120473.0,0.0,111381.0,1.082
2013,1,List(Total renewable energy sources),119980.0,119980.0,0.0,111099.0,1.08
2014,1,List(Total renewable energy sources),118134.0,118134.0,0.0,109103.0,1.083


### 4a. What is the denominator? Test: gross production + imports − exports

In [0]:
gross_supply = yr.select(
    F.col("year").alias("Periods"),
    (F.col("GrossProduction_1") + F.col("ImportsTotal_17") - F.col("ExportsTotal_23")).alias("gross_supply_84575"))

display(per_year.join(gross_supply, "Periods")
        .select("Periods", "max_consumption", "net_consumption_84575", "gross_supply_84575")
        .withColumn("ratio_to_gross_supply", F.round(F.col("max_consumption") / F.col("gross_supply_84575"), 3))
        .orderBy("Periods"))

Periods,max_consumption,net_consumption_84575,gross_supply_84575,ratio_to_gross_supply
2005,118286.0,110301.0,118719.0,0.996
2006,120314.0,111804.0,120294.0,1.0
2007,122718.0,114027.0,122773.0,1.0
2008,123484.0,115266.0,124051.0,0.995
2009,118660.0,109736.0,118391.0,1.002
2010,122094.0,112681.0,120926.0,1.01
2011,123077.0,113555.0,122057.0,1.008
2012,120473.0,111381.0,119614.0,1.007
2013,119980.0,111099.0,119112.0,1.007
2014,118134.0,109103.0,118091.0,1.0


### Findings — Section 4

**Finding 8 — Percentage column is consistent; denominator identified ✅**
- Implied consumption (`_2 / _5 × 100`) agrees across all sources within each year (spread ≤ 0.11%, rounding). Only rows with `_5 ≥ 5%` are used, because `_5` is rounded to one decimal.
- The implied denominator is **not** net consumption: it is ~7–9% higher than `NetConsumptionCalculated_30` in `84575ENG`.
- It matches **gross production + imports − exports** from `84575ENG` within ±0.5% in most years (±1% in 2010–2013). This is the "total electricity consumption" CBS uses for EU-convention percentages.
- Two independent CBS tables reconcile, which supports cross-table consistency.
- **Implication:** when computing renewable shares, use gross production + imports − exports as the denominator to match CBS.

## 5. Validity — are the values physically and logically plausible?
Rules: (a) no negative values, (b) percentages within 0–100, (c) net ≤ gross, (d) capacity factor within 0–1.

In [0]:
def negatives(df, name):
    num_cols = [c for c, t in df.dtypes
                if t in ("int", "bigint", "double") and c not in ("ID", "Periods", "year")]
    row = df.select([F.sum((F.col(c) < 0).cast("int")).alias(c) for c in num_cols]).collect()[0].asDict()
    bad = {c: v for c, v in row.items() if v}
    print(f"{name} — columns with negative values: {bad if bad else 'none'}")

negatives(annual_raw, "ANNUAL")
negatives(monthly, "MONTHLY")

tol = 1  # rounding
display(annual_raw.select(
    F.sum(((F.col("GrossProductionOfElectricity_5") < 0) |
           (F.col("GrossProductionOfElectricity_5") > 100)).cast("int")).alias("pct_out_of_range"),
    F.sum((F.col("NetProductionOfElectricity_3") > F.col("GrossProductionOfElectricity_2") + tol)
          .cast("int")).alias("annual_net_gt_gross"),
))
display(monthly.select(
    F.sum((F.col("NetProductionTotal_3") > F.col("GrossProduction_1") + tol)
          .cast("int")).alias("monthly_net_gt_gross")
))

ANNUAL — columns with negative values: none
MONTHLY — columns with negative values: none


pct_out_of_range,annual_net_gt_gross
0,0


monthly_net_gt_gross
0


### 5a. Capacity factor — production / (capacity × hours)
Production is in million kWh (= GWh), capacity in MW, so CF = production × 1000 / (capacity × hours).
Capacity is reported at **end of year**. In years of rapid growth this overstates the capacity available during the year, so CF is also computed with the average of start- and end-of-year capacity.

In [0]:
from pyspark.sql import Window

cf_sources = ["Onshore wind energy", "Offshore wind energy", "Wind energy, total",
              "Solar photovoltaic", "Hydropower"]
w = Window.partitionBy("EnergySourcesTechniques").orderBy("Periods")

cf = (annual_raw
    .filter(F.col("EnergySourcesTechniques").isin(cf_sources))
    .select("Periods", "EnergySourcesTechniques",
            F.col("GrossProductionOfElectricity_2").alias("prod_GWh"),
            F.col("ElectricalCapacityEndOfYear_8").alias("cap_end"))
    .filter(F.col("cap_end").isNotNull() & (F.col("cap_end") > 0))
    .withColumn("cap_prev", F.lag("cap_end").over(w))
    # previous year missing: 0 if the technology was new, otherwise use end-of-year capacity
    .withColumn("cap_prev", F.when(F.col("cap_prev").isNull() & (F.col("Periods") > 1990), 0)
                             .otherwise(F.coalesce("cap_prev", "cap_end")))
    .withColumn("cap_avg", (F.col("cap_end") + F.col("cap_prev")) / 2)
    .withColumn("hours", F.when(F.col("Periods") % 4 == 0, 8784).otherwise(8760))
    .withColumn("cf_end", F.round(F.col("prod_GWh") * 1000 / (F.col("cap_end") * F.col("hours")), 3))
    .withColumn("cf_avg", F.round(F.col("prod_GWh") * 1000 / (F.col("cap_avg") * F.col("hours")), 3)))

# Summary per source
display(cf.groupBy("EnergySourcesTechniques").agg(
    F.min("cf_end").alias("min_cf_end"), F.max("cf_end").alias("max_cf_end"),
    F.min("cf_avg").alias("min_cf_avg"), F.max("cf_avg").alias("max_cf_avg"),
    F.sum(((F.col("cf_avg") <= 0) | (F.col("cf_avg") >= 1)).cast("int")).alias("cf_out_of_range")))

# Capacity decreases (decommissioning or data issue?)
display(cf.filter(F.col("cap_end") < F.col("cap_prev"))
          .select("EnergySourcesTechniques", "Periods", "cap_prev", "cap_end")
          .orderBy("EnergySourcesTechniques", "Periods"))

EnergySourcesTechniques,min_cf_end,max_cf_end,min_cf_avg,max_cf_avg,cf_out_of_range
Hydropower,0.142,0.437,0.142,0.437,0
Offshore wind energy,0.072,0.441,0.144,0.441,0
Onshore wind energy,0.121,0.304,0.128,0.318,0
Solar photovoltaic,0.0,0.114,0.0,0.118,1
"Wind energy, total",0.121,0.326,0.128,0.347,0


EnergySourcesTechniques,Periods,cap_prev,cap_end
Onshore wind energy,2017,3300.0,3245.0
"Wind energy, total",2017,4257.0,4202.0


In [0]:
display(cf.filter(((F.col("EnergySourcesTechniques") == "Solar photovoltaic") & (F.col("cf_avg") < 0.05)) |
                  ((F.col("EnergySourcesTechniques") == "Hydropower") & (F.col("cf_avg") < 0.2)))
          .select("EnergySourcesTechniques", "Periods", "prod_GWh", "cap_end", "cf_avg")
          .orderBy("EnergySourcesTechniques", "Periods"))

EnergySourcesTechniques,Periods,prod_GWh,cap_end,cf_avg
Hydropower,2011,57.0,37.0,0.176
Hydropower,2017,61.0,37.0,0.188
Hydropower,2020,46.0,37.0,0.142
Hydropower,2022,50.0,38.0,0.15
Hydropower,2025,57.0,38.0,0.171
Solar photovoltaic,1990,0.0,1.0,0.0


### Findings — Section 5

**Finding 9 — Values are physically plausible ✅**
- No negative values in either file; percentages within 0–100; net ≤ gross in both files.
- Capacity factors (average-capacity basis) are within expected ranges: onshore 0.13–0.32, offshore 0.14–0.44, solar ≤ 0.12, hydro 0.14–0.44.
- **End-of-year capacity distorts CF in growth years:** offshore minimum CF is 0.072 with end-of-year capacity vs 0.144 with average capacity (2006, first partial year). **Decision:** use average of start- and end-of-year capacity for CF.
- Solar 1990: production rounds to 0 GWh with 1 MW installed, so CF = 0 is a rounding artefact.
- Onshore capacity decreases once (2017: 3300 → 3245 MW, −1.7%), consistent with decommissioning or repowering.
- Hydro capacity is stable (37–38 MW) while CF varies from 0.14 to 0.44, so variation is driven by water availability. Low-CF years (2011, 2017, 2020, 2022, 2025) are a **hypothesis to test against KNMI precipitation** in the weather-integration phase.

## 6. Cross-file consistency — annual (82610ENG) vs monthly file (84575ENG)
Monthly source columns are **net** production (Finding 5), so they are compared with annual `_3` (net).
Annual rows of the monthly file are used. Tolerance: 1 million kWh.

In [0]:
pairs = {
    "Wind energy, total":   "WindEnergyTotal_12",
    "Onshore wind energy":  "WindEnergyOnShore_13",
    "Offshore wind energy": "WindEnergyOffShore_14",
    "Solar photovoltaic":   "SolarPhotovoltaic_15",
    "Hydropower":           "HydroPower_11",
    "Total biomass":        "Biomass_9",
}

out = []
detail = None
for cat, mcol in pairs.items():
    a = (annual_raw.filter(F.col("EnergySourcesTechniques") == cat)
         .select(F.col("Periods").alias("year"), F.col("NetProductionOfElectricity_3").alias("annual_net")))
    mo = yr.select("year", F.col(mcol).alias("monthly_net"))
    j = (a.join(mo, "year")
          .dropna()
          .withColumn("category", F.lit(cat))
          .withColumn("diff", F.col("monthly_net") - F.col("annual_net"))
          .withColumn("diff_pct", F.round(F.try_divide(F.col("diff"), F.col("annual_net")) * 100, 2)))
    s = j.agg(F.count("*").alias("n"),
              F.count(F.when(F.abs("diff") > 1, True)).alias("failing"),
              F.round(F.max(F.abs("diff_pct")), 2).alias("max_abs_diff_pct")).collect()[0]
    out.append((cat, mcol, s["n"], s["failing"], s["max_abs_diff_pct"]))
    detail = j if detail is None else detail.unionByName(j)

display(spark.createDataFrame(out, ["annual_category", "monthly_column", "years_compared", "failing", "max_abs_diff_pct"]))
display(detail.filter(F.abs("diff") > 1).orderBy("category", "year"))

annual_category,monthly_column,years_compared,failing,max_abs_diff_pct
"Wind energy, total",WindEnergyTotal_12,36,0,0.01
Onshore wind energy,WindEnergyOnShore_13,36,0,0.0
Offshore wind energy,WindEnergyOffShore_14,20,0,0.0
Solar photovoltaic,SolarPhotovoltaic_15,36,1,0.18
Hydropower,HydroPower_11,36,0,0.0
Total biomass,Biomass_9,36,6,1.9


year,annual_net,monthly_net,category,diff,diff_pct
2017,2208.0,2204.0,Solar photovoltaic,-4.0,-0.18
2015,4258.0,4339.0,Total biomass,81.0,1.9
2016,4231.0,4296.0,Total biomass,65.0,1.54
2017,3926.0,3993.0,Total biomass,67.0,1.71
2018,3912.0,3906.0,Total biomass,-6.0,-0.15
2019,5060.0,5067.0,Total biomass,7.0,0.14
2020,7899.0,7891.0,Total biomass,-8.0,-0.1


### Findings — Section 6

**Finding 10 — Annual and monthly files agree for wind and hydro ✅**
- Onshore, offshore and hydro: 0 differences across all years compared (36 / 20 / 36). Wind total: max 0.01%.
- Solar: one year differs (2017: −4 million kWh, −0.18%).
- Biomass: monthly file 1.5–1.9% higher in 2015–2017, within ±0.15% in 2018–2020, matching from 2021. The gap is concentrated in the first years of the full monthly breakdown and disappears later, which suggests revisions applied to the annual table only (not confirmed).
- **Implication:** monthly wind data can be used with confidence for the weather analysis.
- Technical note: Databricks serverless runs in ANSI mode, so division by zero raises an error. `try_divide` is used where the denominator can be 0 (solar 1990).

## Summary — TICKET-002

| # | Check | Result | Key finding |
|---|---|---|---|
| 1 | Hierarchy (components = totals) | ✅ with exception | **Data entry error** in 2023 biogas components (F1); **confirmed by CBS**, correction due Nov 2026 |
| 2 | Time aggregation (months = quarters = years) | ✅ | Tolerance must scale with rounding (F3); total production differs before 1991 (F4); monthly source columns are net (F5) |
| 3 | Normalisation logic (`_1` vs `_2`) | ✅ | CBS documentation confirmed by data (F2, F6); solar 2020 difference explained by CBS as frozen RED reporting values (F7) |
| 4 | Implied consumption (`%` column) | ✅ | Denominator = gross production + imports − exports (F8) |
| 5 | Validity (negatives, %, net ≤ gross, CF) | ✅ | Use average capacity for CF (F9) |
| 6 | Cross-file (annual vs monthly) | ✅ | Wind and hydro match exactly (F10) |

### Decisions carried to TICKET-003 (cleaning)
1. `trim()` all period labels; keep **monthly rows only** from the monthly file.
2. Use **leaf categories** only (no totals) to avoid double counting.
3. Main production measure: **`_2` (gross, not normalised)**; monthly source columns are **net**.
4. Flag 2023 *co-digestion* and *other biogas* as unreliable; no imputation.
5. Treat pre-existence nulls as 0 (CBS: empty cell = not applicable or nil).
6. Capacity factor uses **average of start- and end-of-year capacity**.
7. Mark 2025 (annual) and 2026 (monthly) as **provisional**.
8. Renewable share denominator = gross production + imports − exports (F8).
9. Use `_2 − _1` for wind only with caution: it mixes weather effect and post-submission revisions (F7).

### Data quality verdict
Both datasets are **fit for purpose**. Wind, solar and hydro, the core of the weather analysis, are consistent across hierarchy, time aggregation and files.